In [0]:
# Databricks notebook source
# Czech Fintech — Silver Layer
# Join + typing + SCD Type 2 preparation
# Ultima modifica: 2026-04-03

from pyspark.sql.functions import (
    col, to_date, concat, lit, when, floor, substring,
    year, month, md5, concat_ws, coalesce, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DecimalType, DateType, BooleanType
)
from datetime import datetime

# Config
CATALOG = "czech_fintech"
BRONZE = "bronze"
SILVER = "silver"

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_czech_date(date_str_col):
    """
    Converte YYMMDD (stringa 6 digit) → DateType
    Logica: YY < 50 → 20YY, YY >= 50 → 19YY
    Es: "960215" → 1996-02-15, "000101" → 2000-01-01, "990630" → 1999-06-30
    """
    return to_date(
        when(substring(date_str_col, 1, 2).cast(IntegerType()) < 50,
             concat(lit("20"), date_str_col)
        ).otherwise(
            concat(lit("19"), date_str_col)
        ),
        "yyyyMMdd"
    )

In [0]:


# ============================================================================
# STEP 1: LOAD BRONZE TABLES
# carico le 
# ============================================================================

print("=" * 80)
print("STEP 1: LOADING BRONZE TABLES")
print("=" * 80)


        
trans_bronze = spark.table(f"{CATALOG}.{BRONZE}.transactions")
account_bronze = spark.table(f"{CATALOG}.{BRONZE}.account")
disp_bronze = spark.table(f"{CATALOG}.{BRONZE}.disp")
client_bronze = spark.table(f"{CATALOG}.{BRONZE}.client")
district_bronze = spark.table(f"{CATALOG}.{BRONZE}.district")
card_bronze = spark.table(f"{CATALOG}.{BRONZE}.card")

print(f"✅ transactions: {trans_bronze.count()} rows")
print(f"✅ account: {account_bronze.count()} rows")
print(f"✅ disp: {disp_bronze.count()} rows")
print(f"✅ client: {client_bronze.count()} rows")
print(f"✅ district: {district_bronze.count()} rows")
print(f"✅ card: {card_bronze.count()} rows")


In [0]:

# ============================================================================
# STEP 2: SILVER TRANSACTION
# Join + date parsing + numeric cast + partitioning
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2: BUILDING SILVER TRANSACTIONS")
print("=" * 80)

# Parse date e aggiungi year/month per partition
trans_with_date = trans_bronze.withColumn(
    "date_parsed",
    parse_czech_date(col("date"))
).withColumn(
    "year",
    year(col("date_parsed"))
).withColumn(
    "month",
    month(col("date_parsed"))
)

# Cast numerici
trans_typed = trans_with_date.withColumn(
    "amount", col("amount").cast(DecimalType(15, 2))
).withColumn(
    "balance", col("balance").cast(DecimalType(15, 2))
).withColumn(
    "duration", col("duration").cast(IntegerType())
).withColumn(
    "payments", col("payments").cast(DecimalType(15, 2))
)

# Join con account → disp → client + district
# Logica: account_id → account, account_id → disp (bridge) → client, account_id → district
trans_joined = (
    trans_typed
    .join(account_bronze.select("account_id", "district_id"), "account_id", "left")
    .join(
        disp_bronze.select("account_id", "client_id"),
        ["account_id"],
        "left"
    )
    .join(
        district_bronze.select("district_id", "name"),
        "district_id",
        "left"
    )
    .select(
        trans_typed["*"],
        col("client_id"),
        col("district_id"),
        col("name").alias("district_name"),
        col("date_parsed").alias("date"),
        col("year"),
        col("month"),
        lit(datetime.now().isoformat()).alias("_processed_at")
    )
)

# Rimuovi colonne temporanee
trans_silver = trans_joined.drop("_ingestion_ts", "_source_file", "date_parsed")

print(f"✅ Transactions joined: {trans_silver.count()} rows")

# Salva come tabella Silver
trans_silver.write.mode("overwrite").option("mergeSchema", "true").partitionBy("year", "month").saveAsTable(
    f"{CATALOG}.{SILVER}.transactions"
)
print(f"✅ Tabella {CATALOG}.{SILVER}.transactions creata con partizioni year/month")


In [0]:

# ============================================================================
# STEP 3: SILVER ACCOUNT (pre-SCD Type 2)
# Aggiungi hash per tracking dei cambiamenti
# ============================================================================

print("\n" + "=" * 80)
print("STEP 3: BUILDING SILVER ACCOUNT (pre-SCD Type 2)")
print("=" * 80)

# Calcola hash dei campi principali per SCD Type 2 detection
account_with_hash = account_bronze.withColumn(
    "_bronze_row_hash",
    md5(concat_ws("|", col("frequency")))
).select(
    col("account_id"),
    col("district_id"),
    col("frequency"),
    col("account_created"),
    col("_bronze_row_hash")
)

account_with_hash.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SILVER}.account"
)
print(f"✅ Tabella {CATALOG}.{SILVER}.account creata ({account_with_hash.count()} rows)")


In [0]:

# ============================================================================
# STEP 4: SILVER CLIENT (semplice pass-through, SCD Type 2 ready)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 4: BUILDING SILVER CLIENT")
print("=" * 80)

client_silver = client_bronze.select(
    col("client_id"),
    col("birth_date"),
    col("gender"),
    lit(datetime.now().isoformat()).alias("_processed_at")
)

client_silver.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SILVER}.client"
)
print(f"✅ Tabella {CATALOG}.{SILVER}.client creata ({client_silver.count()} rows)")

# ============================================================================
# STEP 5: SILVER DISTRICT (pass-through)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 5: BUILDING SILVER DISTRICT")
print("=" * 80)

district_silver = district_bronze.select("*", lit(datetime.now().isoformat()).alias("_processed_at"))

district_silver.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SILVER}.district"
)
print(f"✅ Tabella {CATALOG}.{SILVER}.district creata ({district_silver.count()} rows)")


In [0]:

# ============================================================================
# STEP 6: VERIFICATION
# ============================================================================

print("\n" + "=" * 80)
print("VERIFICATION: SILVER LAYER")
print("=" * 80)

expected_silver = {
    "transactions": 40000,
    "account": 4500,
    "client": 5369,
    "district": 77,
}

for table, min_rows in expected_silver.items():
    full_name = f"{CATALOG}.{SILVER}.{table}"
    count = spark.table(full_name).count()
    cols = len(spark.table(full_name).columns)
    print(f"{'✅' if count >= min_rows else '⚠️'} {table:<15} {count:>7} righe  {cols} colonne")

print("\n🎯 Silver layer completato!")
